# Analysis of v8: sample size x lambda (Wu + Berke training, 1 epoch)

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

RUN_FOLDER = Path("runs/train-beta-v8-wu-berke-sample-lrscale-800m")
VALIDATORS = ["kriner2026", "grummon2025"]

In [ ]:
runs = []
for run_dir in sorted(RUN_FOLDER.iterdir()):
    run_info = json.loads((run_dir / "run.json").read_text())
    if run_info["status"] != "finished":
        print(f"skipped (status {run_info['status']}): {run_dir.name}")
        continue

    config = run_info["identity"]["config"]
    sample_sizes = set(config["sample_size_by_dataset"].values())
    if len(sample_sizes) != 1:
        raise ValueError(f"{run_dir.name}: datasets have different sample sizes {sample_sizes}")

    val_history = pd.read_csv(run_dir / "val_history.csv")
    runs.append({
        "run": run_dir.name,
        "lambda": config["beta_lambda"],
        "S": sample_sizes.pop(),
        "val_history": val_history,
    })

print(f"{len(runs)} finished runs")

6 finished runs


## One row per run and validator: the step with this validator's lowest val_loss, and the lowest beta MSE

In [ ]:
def summarize(validator: str) -> pd.DataFrame:
    summary_rows = []
    for run in runs:
        history = run["val_history"]
        validator_rows = history.dropna(subset=[f"{validator}/val_loss"])

        picked_index = validator_rows[f"{validator}/val_loss"].idxmin()
        picked = validator_rows.loc[picked_index]

        lowest_mse_index = validator_rows[f"{validator}/beta_mse"].idxmin()
        lowest_mse = validator_rows.loc[lowest_mse_index]

        summary_rows.append({
            "lambda": run["lambda"],
            "S": run["S"],
            "picked_step": picked["step"],
            "picked_rows_seen": picked["rows_seen"],
            "val_loss": picked[f"{validator}/val_loss"],
            "kl_at_picked": picked[f"{validator}/kl_loss"],
            "beta_mse_at_picked": picked[f"{validator}/beta_mse"],
            "lowest_beta_mse": lowest_mse[f"{validator}/beta_mse"],
            "lowest_beta_mse_step": lowest_mse["step"],
        })
    return pd.DataFrame(summary_rows).sort_values(["S", "lambda"])


summaries = {}
for validator in VALIDATORS:
    summaries[validator] = summarize(validator)
    print(validator)
    display(summaries[validator])

kriner2026


,lambda,S,picked_step,picked_rows_seen,val_loss,kl_at_picked,beta_mse_at_picked,lowest_beta_mse,lowest_beta_mse_step
2,0.00,40,192.0,7680.0,0.632487,0.632487,0.004002,0.003378,640.0
0,0.25,40,288.0,11520.0,0.637461,0.633086,0.004375,0.003448,800.0
4,1.00,40,928.0,37060.0,0.645938,0.632419,0.003380,0.003305,992.0
3,0.00,160,128.0,20350.0,0.627504,0.627504,0.003714,0.003244,56.0
1,0.25,160,128.0,20350.0,0.633555,0.629932,0.003623,0.003542,160.0
5,1.00,160,168.0,26750.0,0.651017,0.634592,0.004106,0.003661,144.0


grummon2025


,lambda,S,picked_step,picked_rows_seen,val_loss,kl_at_picked,beta_mse_at_picked,lowest_beta_mse,lowest_beta_mse_step
2,0.00,40,800.0,31940.0,0.814549,0.814549,0.002434,0.002299,640.0
0,0.25,40,480.0,19179.0,0.847053,0.842979,0.003148,0.003148,480.0
4,1.00,40,480.0,19179.0,0.820403,0.808855,0.002231,0.002118,640.0
3,0.00,160,120.0,19109.0,0.874356,0.874356,0.006179,0.003245,160.0
1,0.25,160,160.0,25470.0,0.867848,0.862908,0.003817,0.003817,160.0
5,1.00,160,160.0,25470.0,0.835554,0.826328,0.001782,0.001782,160.0


## 2x2 helper: rows = sample size S, columns = lambda; one table per validator

In [ ]:
def show_two_by_two(column: str, decimals: int) -> None:
    for validator in VALIDATORS:
        table = summaries[validator].pivot(index="S", columns="lambda", values=column)
        print(validator)
        display(table.round(decimals))

## val_loss (lowest; = KL + lambda x std beta MSE, so compare within a lambda column)

In [ ]:
show_two_by_two("val_loss", decimals=4)

kriner2026


lambda,0.00,0.25,1.00
S,,,
40,0.6325,0.6375,0.6459
160,0.6275,0.6336,0.6510


grummon2025


lambda,0.00,0.25,1.00
S,,,
40,0.8145,0.8471,0.8204
160,0.8744,0.8678,0.8356


## KL at the picked step (comparable across lambda)

In [ ]:
show_two_by_two("kl_at_picked", decimals=4)

kriner2026


lambda,0.00,0.25,1.00
S,,,
40,0.6325,0.6331,0.6324
160,0.6275,0.6299,0.6346


grummon2025


lambda,0.00,0.25,1.00
S,,,
40,0.8145,0.8430,0.8089
160,0.8744,0.8629,0.8263


## Lowest beta MSE over the epoch

In [ ]:
show_two_by_two("lowest_beta_mse", decimals=5)

kriner2026


lambda,0.00,0.25,1.00
S,,,
40,0.00338,0.00345,0.00331
160,0.00324,0.00354,0.00366


grummon2025


lambda,0.00,0.25,1.00
S,,,
40,0.00230,0.00315,0.00212
160,0.00324,0.00382,0.00178


## Beta MSE at the picked step (for kriner2026 this is the checkpoint the run keeps)

In [ ]:
show_two_by_two("beta_mse_at_picked", decimals=5)

kriner2026


lambda,0.00,0.25,1.00
S,,,
40,0.00400,0.00438,0.00338
160,0.00371,0.00362,0.00411


grummon2025


lambda,0.00,0.25,1.00
S,,,
40,0.00243,0.00315,0.00223
160,0.00618,0.00382,0.00178
